# Day 24: Zero-Downtime Deployment & Cost Estimation
> *Inference Engineering* — Chapter 7.4 | Philip Kiely, Baseten Books 2026

**Layer:** Infrastructure | **Prerequisite:** Day 23 (Multi-Cloud), Day 21 (Autoscaling)


## Concept Overview

Zero-downtime deployment is required for production inference services — users should not experience errors or elevated latency during model updates. Blue-green deployment maintains two identical environments (blue=current, green=new), switches traffic atomically after validation. Canary deployment gradually shifts traffic from old to new version, monitoring error rates and latency. Cost estimation is the other half of production operations: accurate $/token modeling enables pricing decisions and capacity planning.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from dataclasses import dataclass
from typing import Dict, List

print('Zero-downtime deployment and cost estimation demo')


## 1. Blue-Green Deployment

Two environments: blue (current) and green (new). Traffic switches atomically after health checks pass.


In [ ]:
@dataclass
class DeploymentEnvironment:
    name: str
    version: str
    health: float  # 0-1
    latency_p99_ms: float
    error_rate: float  # 0-1
    active_requests: int = 0

class BlueGreenDeployment:
    def __init__(self):
        self.blue = DeploymentEnvironment('blue', 'v1.0', 1.0, 120, 0.001)
        self.green = DeploymentEnvironment('green', 'v2.0', 1.0, 110, 0.001)
        self.active = 'blue'
        self.traffic_to_green = 0.0
        self.events = []

    def deploy_new_version(self, new_version):
        self.green.version = new_version
        self.green.health = 0.0  # starting up
        self.events.append(f'[t=0] Deploying {new_version} to green')

    def warmup_green(self, t):
        """Simulate green environment warming up."""
        self.green.health = min(1.0, t / 30.0)  # 30s warmup
        if self.green.health >= 1.0 and self.traffic_to_green == 0:
            self.events.append(f'[t={t:.0f}s] Green healthy, starting traffic shift')

    def run_health_checks(self):
        """Validate green before full cutover."""
        checks = [
            ('HTTP health endpoint', self.green.health > 0.9),
            ('Error rate < 1%', self.green.error_rate < 0.01),
            ('P99 latency < 200ms', self.green.latency_p99_ms < 200),
            ('GPU memory stable', True),
        ]
        all_pass = all(v for _, v in checks)
        print('Green environment health checks:')
        for name, passed in checks:
            status = '✓ PASS' if passed else '✗ FAIL'
            print(f'  {name:<35} {status}')
        return all_pass

    def switch_traffic(self, fraction):
        """Gradually shift traffic to green."""
        self.traffic_to_green = fraction
        self.events.append(f'Traffic to green: {fraction:.0%}')

    def full_cutover(self):
        self.active = 'green'
        self.traffic_to_green = 1.0
        self.events.append('Full cutover to green complete')

deploy = BlueGreenDeployment()
deploy.deploy_new_version('v2.1')

# Simulate warmup
for t in [5, 10, 20, 30, 35]:
    deploy.warmup_green(t)

passed = deploy.run_health_checks()
if passed:
    print('\nCanary rollout:')
    for frac in [0.05, 0.10, 0.25, 0.50, 1.0]:
        deploy.switch_traffic(frac)
        print(f'  {frac:.0%} traffic to green (v2.1)')
    deploy.full_cutover()

print('\nDeployment event log:')
for event in deploy.events:
    print(f'  {event}')


## 2. Canary Deployment with Error Rate Monitoring

Canary releases gradually shift traffic while monitoring error rates and latency to catch regressions early.


In [ ]:
class CanaryDeployment:
    def __init__(self, error_threshold=0.01, latency_threshold_ms=200):
        self.canary_traffic = 0.0
        self.error_threshold = error_threshold
        self.latency_threshold_ms = latency_threshold_ms
        self.metrics_history = []

    def simulate_step(self, canary_pct, inject_error=False):
        """Simulate one monitoring window."""
        # Baseline error rate ~0.1%, canary might have regression
        baseline_err = np.random.beta(1, 999)
        canary_err = np.random.beta(3, 97) if inject_error else np.random.beta(1, 999)
        # Blended metrics
        blended_err = (1 - canary_pct) * baseline_err + canary_pct * canary_err
        canary_latency = 115 + (50 if inject_error else 0) + np.random.normal(0, 5)
        self.metrics_history.append({
            'canary_pct': canary_pct,
            'error_rate': blended_err,
            'canary_latency': canary_latency,
        })
        return blended_err < self.error_threshold and canary_latency < self.latency_threshold_ms

# Normal rollout
canary = CanaryDeployment()
print('Normal canary rollout (no regression):')
for pct in [0.01, 0.05, 0.1, 0.25, 0.5, 1.0]:
    ok = canary.simulate_step(pct)
    m = canary.metrics_history[-1]
    print(f'  {pct:>5.0%} → err={m["error_rate"]*100:.2f}% lat={m["canary_latency"]:.0f}ms {"OK" if ok else "ROLLBACK"}')

# Rollout with regression at 10%
canary2 = CanaryDeployment()
print('\nCanary rollout with regression injected at 10%:')
for i, pct in enumerate([0.01, 0.05, 0.1, 0.25, 0.5, 1.0]):
    inject = i >= 2  # inject regression from 10% onwards
    ok = canary2.simulate_step(pct, inject_error=inject)
    m = canary2.metrics_history[-1]
    status = 'OK' if ok else 'ROLLBACK!'
    print(f'  {pct:>5.0%} → err={m["error_rate"]*100:.2f}% lat={m["canary_latency"]:.0f}ms {status}')
    if not ok:
        print('  ⚠ Rolling back to stable version')
        break


## 3. Cost per Token Estimation

The fundamental unit cost for an inference service: $/token, factoring in GPU cost, utilization, and throughput.


In [ ]:
def cost_per_token(gpu_cost_hr, gpu_utilization, tokens_per_second_per_gpu,
                  overhead_pct=0.3):
    """
    Compute cost per 1M tokens.
    overhead_pct: fraction for storage, network, CPU, etc.
    """
    effective_tps = tokens_per_second_per_gpu * gpu_utilization
    tokens_per_hour = effective_tps * 3600
    cost_per_token = gpu_cost_hr / tokens_per_hour
    total_with_overhead = cost_per_token * (1 + overhead_pct)
    return total_with_overhead * 1e6  # per 1M tokens

scenarios = [
    # (label, gpu_cost_hr, tps_per_gpu, utilization)
    ('A100 on AWS (on-demand)', 3.5, 500, 0.7),
    ('H100 on AWS (on-demand)', 10.0, 2000, 0.7),
    ('H100 on AWS (reserved)', 7.0, 2000, 0.8),
    ('DGX Spark (on-prem, amortized)', 0.5, 300, 0.8),  # ~$50/day amortized
    ('H100 on Lambda Labs', 7.5, 2000, 0.75),
    ('B200 on AWS (projected)', 18.0, 5000, 0.75),
]

print(f'{"Scenario":<40} {"$/1M tokens":>12}')
print('-' * 55)
for label, cost_hr, tps, util in scenarios:
    cost = cost_per_token(cost_hr, util, tps)
    print(f'{label:<40} ${cost:>11.3f}')

print('\nFor context:')
print('  OpenAI GPT-4o:  ~$5/1M input, $15/1M output')
print('  Claude 3.5 Sonnet: ~$3/1M input, $15/1M output')
print('  Self-hosted 70B: $0.5-2/1M tokens')


## Experiments: Try These

1. **Implement a health check**: Write a `/health` endpoint for a vLLM server that checks GPU memory, queue depth, and model loaded status. Return 503 if unhealthy.
2. **Cost dashboard**: Build a cost tracking tool that logs tokens processed per minute and accumulates $/token in real time.
3. **Shadow deployment**: Implement a shadow deployment where 100% of traffic goes to blue but a copy goes to green (for testing). Compare response differences.


## Key Takeaways

- Blue-green deployment maintains two environments, switches traffic atomically after green passes all health checks — zero request failure during update.
- Canary deployment gradually shifts traffic while monitoring error rate and latency — automatically rolls back on regression, stopping impact at the canary percentage.
- Cost per token = GPU cost / (tokens per second × utilization × 3600) × overhead — on-prem at 80% utilization can be 3-5x cheaper than cloud at 70%.
- Infrastructure-layer decisions (MIG, colocation, reserved capacity) have a larger impact on $/token than optimization-layer decisions (quantization, batching) at scale.

## References
- *Inference Engineering* Ch 7.4 — Philip Kiely, Baseten Books 2026
- Martin Fowler, "BlueGreenDeployment" (martinfowler.com)
- Google SRE Book, Chapter 27: Reliable Product Launches
